# Insurance Stacking Regression Detailed EDA

In [1]:
import pandas as pd
df=pd.read_csv('../data/insurance.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,56,male,30.0,1,yes,southeast,49427.95
1,46,male,27.7,3,no,northwest,23347.30
2,32,female,27.2,1,yes,southeast,45630.20
3,60,female,23.3,4,no,southeast,26642.05
4,25,male,35.2,4,no,northwest,21021.26


In [2]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       300 non-null    int64  
 1   sex       300 non-null    object 
 2   bmi       300 non-null    float64
 3   children  300 non-null    int64  
 4   smoker    300 non-null    object 
 5   region    300 non-null    object 
 6   charges   300 non-null    float64
dtypes: float64(2), int64(2), object(3)
memory usage: 16.5+ KB


age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [3]:
from sklearn.preprocessing import LabelEncoder
for c in df.select_dtypes(include='object').columns:
    df[c]=LabelEncoder().fit_transform(df[c])

In [4]:
from sklearn.model_selection import train_test_split
X=df.drop('charges',axis=1)
y=df['charges']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rf = RandomForestRegressor(random_state=42)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(rf, param_grid, cv=5, scoring='r2', n_jobs=-1)
rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
print("Best params:", rf_grid.best_params_)
print("Best CV R2:", rf_grid.best_score_)

Best params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
Best CV R2: 0.9053034782372679


In [6]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=42)
param_grid = {
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt_grid = GridSearchCV(dt, param_grid, cv=5, scoring='r2', n_jobs=-1)
dt_grid.fit(X_train, y_train)

best_dt = dt_grid.best_estimator_
print("Best params:", dt_grid.best_params_)
print("Best CV R2:", dt_grid.best_score_)

Best params: {'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 2}
Best CV R2: 0.869501875929874


In [7]:
dt = DecisionTreeRegressor(random_state=42)
param_grid = {
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt_grid = GridSearchCV(dt, param_grid, cv=5, scoring='r2', n_jobs=-1)
dt_grid.fit(X_train, y_train)

best_dt = dt_grid.best_estimator_
print("Best params:", dt_grid.best_params_)
print("Best CV R2:", dt_grid.best_score_)

Best params: {'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 2}
Best CV R2: 0.869501875929874


In [8]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV

stack = StackingRegressor(
    estimators=[('rf', best_rf), ('dt', best_dt)],
    final_estimator=RidgeCV(),
    cv=5,
    n_jobs=-1
)

stack.fit(X_train, y_train)
best_stack = stack

print("Stacking regressor fitted")
print("Train R2:", best_stack.score(X_train, y_train))

Stacking regressor fitted
Train R2: 0.9781139190544872


In [10]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# R2, MAE, MSE, RMSE Evaluation

def eval_model(name, model, X, y):
    y_pred = model.predict(X)
    print(f"{name} R2: {r2_score(y, y_pred):.4f}")
    print(f"{name} MAE: {mean_absolute_error(y, y_pred):.4f}")
    print(f"{name} MSE: {mean_squared_error(y, y_pred):.4f}")
    print(f"{name} RMSE: {mean_squared_error(y, y_pred):.4f}")
    print()

for name, model in [
    ('RandomForest', best_rf),
    ('DecisionTree', best_dt),
    ('Stacking', best_stack)
]:
    eval_model(name, model, X_test, y_test)

RandomForest R2: 0.9122
RandomForest MAE: 2657.9313
RandomForest MSE: 9721971.0052
RandomForest RMSE: 9721971.0052

DecisionTree R2: 0.8440
DecisionTree MAE: 3576.9266
DecisionTree MSE: 17271234.6478
DecisionTree RMSE: 17271234.6478

Stacking R2: 0.9168
Stacking MAE: 2524.0514
Stacking MSE: 9206178.6249
Stacking RMSE: 9206178.6249

